### Dataset Formation

In [76]:
import os
import torch
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T

class MyRatDataset(Dataset):
    """
    A dataset that reads images and YOLO-format annotations from a folder.
    Expected structure:
        root/
          images/
            img1.jpg
            img2.jpg
            ...
          labels/
            img1.txt
            img2.txt
            ...
    Each label file should contain lines (YOLO format):
        class_id  x_center_norm  y_center_norm  width_norm  height_norm
    Negative samples can be created on the fly with a given probability.
    This updated version prints a warning and ignores any image whose label file contains negative values.
    """
    def __init__(self, root, transforms=None, negative_sample_ratio=0.3):
        """
        Args:
            root (str): Path to the subset folder (e.g., 'new_dataset/train').
                        Must contain 'images/' and 'labels/' subfolders.
            transforms (callable, optional): Transformations to apply to the PIL image.
            negative_sample_ratio (float): The probability (0 to 1) of generating a negative sample from a positive image.
        """
        self.root = root
        self.transforms = transforms
        self.negative_sample_ratio = negative_sample_ratio
        
        self.img_dir = os.path.join(root, "images")
        self.lbl_dir = os.path.join(root, "labels")
        
        # Collect image file names
        self.imgs = [f for f in os.listdir(self.img_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.imgs.sort()  # for consistent ordering

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # Load image
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        # Load annotations from the corresponding label file
        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.lbl_dir, label_name)
        
        boxes = []
        labels = []
        corrupt = False  # flag for negative values in the label file
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    try:
                        class_id = int(parts[0]) + 1  # shift class indices if needed
                        x_center_norm = float(parts[1])
                        y_center_norm = float(parts[2])
                        width_norm    = float(parts[3])
                        height_norm   = float(parts[4])
                    except ValueError:
                        continue

                    # Check for negative values
                    if x_center_norm < 0 or y_center_norm < 0 or width_norm < 0 or height_norm < 0:
                        # print(f"WARNING {img_path}: ignoring corrupt image/label: negative label values "
                        #       f"[{x_center_norm:.5f}, {y_center_norm:.5f}, {width_norm:.5f}, {height_norm:.5f}]")
                        corrupt = True
                        break

                    # Convert normalized coords to absolute pixel coordinates
                    x_center = x_center_norm * w
                    y_center = y_center_norm * h
                    box_width = width_norm * w
                    box_height = height_norm * h
                    x_min = x_center - box_width / 2
                    y_min = y_center - box_height / 2
                    x_max = x_center + box_width / 2
                    y_max = y_center + box_height / 2
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(class_id)
        
        # If corrupt, treat this image as a negative sample (no boxes)
        if corrupt:
            boxes = []
            labels = []

        # If no annotations, treat image as a negative sample naturally
        if len(boxes) == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Optionally generate a negative sample from a positive image
        if boxes.shape[0] > 0 and random.random() < self.negative_sample_ratio:
            img = self.generate_negative_sample(img, boxes)
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        
        image_id = torch.tensor([idx])
        if boxes.size(0) > 0:
            area = (boxes[:,2] - boxes[:,0]) * (boxes[:,3] - boxes[:,1])
        else:
            area = torch.empty((0,), dtype=torch.float32)
        iscrowd = torch.zeros((labels.shape[0],), dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }
        
        if self.transforms:
            img = self.transforms(img)
        
        return img, target

    def generate_negative_sample(self, img, boxes):
        """
        Generate a negative sample by filling the regions of each bounding box in the image with random noise.
        Args:
            img (PIL.Image): The original image.
            boxes (Tensor): Tensor of shape [N, 4] with bounding box coordinates in absolute pixel values.
        Returns:
            PIL.Image: The image with the rat regions replaced with noise.
        """ 
        img_np = np.array(img)
        for box in boxes:
            x_min, y_min, x_max, y_max = box.int().tolist()
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_np.shape[1], x_max)
            y_max = min(img_np.shape[0], y_max)
            if x_max > x_min and y_max > y_min:
                noise = np.random.randint(0, 256, (y_max - y_min, x_max - x_min, 3), dtype=np.uint8)
                img_np[y_min:y_max, x_min:x_max, :] = noise
        return Image.fromarray(img_np)

# Example usage:
if __name__ == "__main__":
    transforms = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    
    dataset_root = "new_dataset\\train"  # Folder with subfolders: images/ and labels/
    dataset = MyRatDataset(root=dataset_root, transforms=transforms, negative_sample_ratio=0.3)
    
    # Inspect a sample
    img, target = dataset[0]
    print("Sample target:", target)
    
    from torch.utils.data import DataLoader
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))
    
    for images, targets in dataloader:
        print(f"Batch has {len(images)} images")
        for t in targets:
            print("Boxes:", t["boxes"], "Labels:", t["labels"])
        break


Sample target: {'boxes': tensor([], size=(0, 4)), 'labels': tensor([], dtype=torch.int64), 'image_id': tensor([0]), 'area': tensor([]), 'iscrowd': tensor([], dtype=torch.int64)}
Batch has 4 images
Boxes: tensor([], size=(0, 4)) Labels: tensor([], dtype=torch.int64)
Boxes: tensor([], size=(0, 4)) Labels: tensor([], dtype=torch.int64)
Boxes: tensor([[476.0000, 126.9998, 608.0000, 393.9998]]) Labels: tensor([1])
Boxes: tensor([[319.0003, 125.0003, 724.0000, 529.0002]]) Labels: tensor([1])


In [77]:
import matplotlib.pyplot as plt
import numpy as np

def show_negative_samples(dataset, dataset_name="Dataset"):
    print(f"\nShowing negative samples from: {dataset_name}")
    count = 0
    
    for i in range(len(dataset)):
        img, target = dataset[i]
        
        # Check if there are no bounding boxes (i.e., negative sample)
        if target["boxes"].numel() == 0:
            count += 1
            print(f"  Negative sample index: {i}, image file: {dataset.imgs[i]}")
            
            # Convert the tensor image to NumPy for display
            # If your dataset uses normalization, you may want to 'undo' it for visualization
            img_np = img.permute(1, 2, 0).cpu().numpy()  # [H, W, C]
            
            # Optional: undo normalization for better display
            # mean = np.array([0.485, 0.456, 0.406])
            # std = np.array([0.229, 0.224, 0.225])
            # img_np = std * img_np + mean
            # img_np = np.clip(img_np, 0, 1)
            
            plt.figure(figsize=(6, 6))
            plt.imshow(img_np)
            plt.title(f"{dataset_name} Negative Sample idx={i}")
            plt.axis('off')
            plt.show()
    
    print(f"Found {count} negative samples in {dataset_name}.")


## Train

In [78]:
import os
import torch
import torchvision
import torchvision.transforms as T
import torchvision.transforms.v2 as T2
from torch.utils.data import DataLoader
from torchsummary import summary
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import CosineAnnealingLR  # Changed to cosine annealing scheduler

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


# Assume your custom dataset class MyRatDataset is defined elsewhere.
# It should return images (PIL Images) and targets (a dict with keys "boxes", "labels", etc.)

# Define transforms: converting image to tensor and normalizing.
transforms = T.Compose([
    # T.RandomVerticalFlip(p=0.5),
    # T.RandomHorizontalFlip(p=0.0),      # 0% chance to flip horizontally
    # T.RandomResizedCrop(224),           # Random scale and crop
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Create training and validation datasets and dataloaders.
# train_dataset = MyRatDataset(root="new_dataset\\train", transforms=transforms, negative_sample_ratio=0.3)#0.3
# val_dataset   = MyRatDataset(root="new_dataset\\valid", transforms=transforms, negative_sample_ratio=0.15)#0.0
train_dataset = MyRatDataset(root="new_dataset\\train", transforms=transforms, negative_sample_ratio=0.01)#0.3
val_dataset   = MyRatDataset(root="new_dataset\\valid", transforms=transforms, negative_sample_ratio=0.01)#0.0

# Count negatives in training set: only count negatives when a label file exists.
num_positive = 0
num_negative = 0
print("Scanning training dataset...")
for i in range(len(train_dataset)):
    _, target = train_dataset[i]
    label_name = os.path.splitext(train_dataset.imgs[i])[0] + ".txt"
    label_path = os.path.join(train_dataset.lbl_dir, label_name)
    if os.path.exists(label_path):
        # File exists: count empty (negative) or non-empty (positive) based on target.
        if len(target["boxes"]) == 0:
            num_negative += 1
        else:
            num_positive += 1
    else:
        print(f'{label_path} doesnt exsists')
        pass
print(f"✅ Total train_dataset: {len(dataset)}")
print(f"🟥 Positive train_dataset (with boxes): {num_positive}")
print(f"🟦 Negative train_dataset (no boxes): {num_negative}")

# Counters for samples
num_positive = 0
num_negative = 0
print("Scanning val_dataset dataset...")
for i in range(len(val_dataset)):
    _, target = val_dataset[i]
    label_name = os.path.splitext(val_dataset.imgs[i])[0] + ".txt"
    label_path = os.path.join(val_dataset.lbl_dir, label_name)
    if os.path.exists(label_path):
        # File exists: count empty (negative) or non-empty (positive) based on target.
        if len(target["boxes"]) == 0:
            num_negative += 1
        else:
            num_positive += 1
    else:
        print(f'{label_path} doesnt exsists')
        pass
print(f"✅ Total val_dataset: {len(val_dataset)}")
print(f"🟥 Positive val_dataset (with boxes): {num_positive}")
print(f"🟦 Negative val_dataset (no boxes): {num_negative}")


Scanning training dataset...
new_dataset\train\labels\desk 12.txt doesnt exsists
new_dataset\train\labels\desk 7.txt doesnt exsists
✅ Total train_dataset: 1402
🟥 Positive train_dataset (with boxes): 1019
🟦 Negative train_dataset (no boxes): 381
Scanning val_dataset dataset...
✅ Total val_dataset: 600
🟥 Positive val_dataset (with boxes): 433
🟦 Negative val_dataset (no boxes): 167


In [79]:
# show_negative_samples(train_dataset, "Train")
# show_negative_samples(val_dataset, "Val")

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=lambda batch: tuple(zip(*batch))
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda batch: tuple(zip(*batch))
)

# Load the SSDLite model with COCO pre-trained weights for the detection head and 2 classes (background and rat).
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,  # 2 classes: background and rat
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)

# Move model to device.
device = torch.device("cuda:0")
model.to(device)

# Define a helper to freeze all BatchNorm layers.
def freeze_bn(module):
    if isinstance(module, torch.nn.BatchNorm2d):
        module.eval()
        for param in module.parameters():
            param.requires_grad = False

# Freeze all BN layers in the model.
model.apply(freeze_bn)

# Create optimizer using AdamW with the specified learning rate and weight decay.
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.0005,
    momentum=0.9,
    weight_decay=0.005
)

# Use CosineAnnealingLR scheduler.
num_epochs = 250  # (Adjust as needed; the command-line example used 660 epochs)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

# Set up AMP scaler and TensorBoard writer.
scaler = GradScaler()
writer = SummaryWriter(log_dir=r"C:\Machine Learning\Rat Tracking using TensorFlow\runs")

train_losses = []
val_losses = []
train_cls_losses = []
val_cls_losses = []
train_bbox_losses = []
val_bbox_losses = []

# Optionally, print a model summary.
summary(model, input_size=(3, 320, 320))

best_val_loss = float('inf')  # Initialize best validation loss.
best_epoch = -1

try:
    # -----------------------------
    # Training and Validation Loop
    # -----------------------------
    for epoch in range(num_epochs):
        model.train()  # Set detection head to train mode.
        model.apply(freeze_bn)  # Re-freeze all BN layers.
        
        epoch_train_loss = 0.0
        epoch_train_cls_loss = 0.0
        epoch_train_bbox_loss = 0.0
        
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Filter out samples with empty bounding boxes.
            filtered_images = []
            filtered_targets = []
            for img, tgt in zip(images, targets):
                if tgt["boxes"].numel() > 0:
                    filtered_images.append(img)
                    filtered_targets.append(tgt)
            
            optimizer.zero_grad()
            with autocast():
                if len(filtered_images) == 0:
                    total_loss = torch.tensor(0., device=device, requires_grad=True)
                else:
                    loss_dict = model(filtered_images, filtered_targets)
                    cls_loss = loss_dict["classification"]
                    bbox_loss = loss_dict["bbox_regression"]
                    total_loss = cls_loss + bbox_loss
            
            # Only perform backward if loss is nonzero.
            if total_loss.item() != 0:
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            
            epoch_train_loss += total_loss.item()
            if len(filtered_images) > 0:
                epoch_train_cls_loss += cls_loss.item()
                epoch_train_bbox_loss += bbox_loss.item()
        
        epoch_train_loss /= len(train_loader)
        epoch_train_cls_loss /= len(train_loader)
        epoch_train_bbox_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        train_cls_losses.append(epoch_train_cls_loss)
        train_bbox_losses.append(epoch_train_bbox_loss)
        
        writer.add_scalar("Loss/Train/Total", epoch_train_loss, epoch)
        writer.add_scalar("Loss/Train/Class", epoch_train_cls_loss, epoch)
        writer.add_scalar("Loss/Train/BBox", epoch_train_bbox_loss, epoch)
        print(
            f"Epoch {epoch+1}/{num_epochs}: "
            f"Train Total Loss: {epoch_train_loss:.4f}, "
            f"Train cls Loss: {epoch_train_cls_loss:.4f}, "
            f"Train bbox Loss: {epoch_train_bbox_loss:.4f}"
        )
        
        # ------------------
        # Validation Loop
        # ------------------
        model.train()  # SSDLite returns losses only in train mode.
        model.apply(freeze_bn)
        
        epoch_val_loss = 0.0
        epoch_val_cls_loss = 0.0
        epoch_val_bbox_loss = 0.0
        
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                filtered_images = []
                filtered_targets = []
                for img, tgt in zip(images, targets):
                    if tgt["boxes"].numel() > 0:
                        filtered_images.append(img)
                        filtered_targets.append(tgt)
                
                if len(filtered_images) == 0:
                    total_loss = torch.tensor(0., device=device)
                else:
                    loss_dict = model(filtered_images, filtered_targets)
                    cls_loss = loss_dict["classification"]
                    bbox_loss = loss_dict["bbox_regression"]
                    total_loss = cls_loss + bbox_loss
                
                epoch_val_loss += total_loss.item()
                if len(filtered_images) > 0:
                    epoch_val_cls_loss += cls_loss.item()
                    epoch_val_bbox_loss += bbox_loss.item()
        
        epoch_val_loss /= len(val_loader)
        epoch_val_cls_loss /= len(val_loader)
        epoch_val_bbox_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        val_cls_losses.append(epoch_val_cls_loss)
        val_bbox_losses.append(epoch_val_bbox_loss)
        
        writer.add_scalar("Loss/Val/Total", epoch_val_loss, epoch)
        writer.add_scalar("Loss/Val/Class", epoch_val_cls_loss, epoch)
        writer.add_scalar("Loss/Val/BBox", epoch_val_bbox_loss, epoch)
        
        print(
            f"Epoch {epoch+1}/{num_epochs}: "
            f"Val Total Loss: {epoch_val_loss:.4f}, "
            f"Val cls Loss: {epoch_val_cls_loss:.4f}, "
            f"Val bbox Loss: {epoch_val_bbox_loss:.4f}"
        )
        
        scheduler.step()  # CosineAnnealingLR step after each epoch.
        
        # Save the best model (based on validation loss)
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "Best_Model.pth")
            print(f"Best model saved at epoch {best_epoch} with val loss {best_val_loss:.4f}")
    
except KeyboardInterrupt:
    print("Training interrupted. Saving the best model so far...")

# Save the final model state (optional).
torch.save(model.state_dict(), "Final_Model.pth")
print(f"Final model saved to Final_Model.pth")
print(f"Best model was from epoch {best_epoch} with val loss {best_val_loss:.4f}")


C:\Users\mzarrar\AppData\Local\Temp\ipykernel_2104\407195977.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Layer (type:depth-idx)                             Param #
├─SSDLiteFeatureExtractorMobileNet: 1-1            --
|    └─Sequential: 2-1                             --
|    |    └─Sequential: 3-1                        (869,096)
|    |    └─Sequential: 3-2                        (2,102,856)
|    └─ModuleList: 2-2                             --
|    |    └─Sequential: 3-3                        (381,184)
|    |    └─Sequential: 3-4                        (100,480)
|    |    └─Sequential: 3-5                        (67,712)
|    |    └─Sequential: 3-6                        (25,664)
├─DefaultBoxGenerator: 1-2                         --
├─SSDLiteHead: 1-3                                 --
|    └─SSDLiteClassificationHead: 2-3              --
|    |    └─ModuleList: 3-7                        (64,104)
|    └─SSDLiteRegressionHead: 2-4                  --
|    |    └─ModuleList: 3-8                        (97,584)
├─GeneralizedRCNNTransform: 1-4                    --
Total params: 3,708,680

Epoch 1/250:   0%|          | 0/88 [00:00<?, ?it/s]C:\Users\mzarrar\AppData\Local\Temp\ipykernel_2104\407195977.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/250: Train Total Loss: 5.7933, Train cls Loss: 2.8489, Train bbox Loss: 2.9444
Epoch 1/250: Val Total Loss: 4.9371, Val cls Loss: 2.1340, Val bbox Loss: 2.8032
Best model saved at epoch 1 with val loss 4.9371


Epoch 2/250: Train Total Loss: 5.7034, Train cls Loss: 2.7745, Train bbox Loss: 2.9289
Epoch 2/250: Val Total Loss: 4.9022, Val cls Loss: 2.1095, Val bbox Loss: 2.7927
Best model saved at epoch 2 with val loss 4.9022


...

Epoch 215/250: Train Total Loss: 1.8251, Train cls Loss: 1.3438, Train bbox Loss: 0.4813
Epoch 215/250: Val Total Loss: 2.8518, Val cls Loss: 1.6808, Val bbox Loss: 1.1710


Epoch 216/250: Train Total Loss: 1.8195, Train cls Loss: 1.3422, Train bbox Loss: 0.4772
Epoch 216/250: Val Total Loss: 2.8791, Val cls Loss: 1.6925, Val bbox Loss: 1.1865


Epoch 217/250: Train Total Loss: 1.8262, Train cls Loss: 1.3446, Train bbox Loss: 0.4816
Epoch 217/250: Val Total Loss: 2.8634, Val cls Loss: 1.6924, Val bbox Loss: 1.1710


Epoch 218/250: Train Total Loss: 1.8140, Train cls Loss: 1.3425, Train bbox Loss: 0.4716
Epoch 218/250: Val Total Loss: 2.8646, Val cls Loss: 1.6941, Val bbox Loss: 1.1705


Epoch 219/250: Train Total Loss: 1.8298, Train cls Loss: 1.3465, Train bbox Loss: 0.4833


In [ ]:
### PLot
import csv

# Save losses to a CSV file
csv_file = "losses.csv"
with open(csv_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Epoch", "Train Loss", "Train cls Loss", "Train bbox Loss", "Val Loss", "Val cls Loss", "Val bbox Loss",])
    for epoch, t_loss, t_cls, t_bbox, v_loss, v_cls, v_bbox in zip(range(1, num_epochs + 1), train_losses, train_cls_losses, train_bbox_losses, val_losses, val_cls_losses, val_bbox_losses):
        writer.writerow([epoch, t_loss, t_cls, t_bbox, v_loss, v_cls, v_bbox])
print(f"Losses saved to {csv_file}")

: 

In [ ]:
import csv
import matplotlib.pyplot as plt

epochs = []
train_cls_losses = []
val_cls_losses = []
train_bbox_losses = []
val_bbox_losses = []

# Load data from the CSV file
with open("losses.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row["Epoch"]))
        train_cls_losses.append(float(row["Train cls Loss"]))
        val_cls_losses.append(float(row["Val cls Loss"]))
        train_bbox_losses.append(float(row["Train bbox Loss"]))
        val_bbox_losses.append(float(row["Val bbox Loss"]))

# Plot the loss curves with consistent colors and line styles
cls_color = 'blue'
bbox_color = 'red'

plt.plot(epochs, train_cls_losses, color=cls_color, linestyle='-', label="Train CLS Loss")
plt.plot(epochs, val_cls_losses, color=cls_color, linestyle=':', label="Val CLS Loss")
plt.plot(epochs, train_bbox_losses, color=bbox_color, linestyle='-', label="Train BBOX Loss")
plt.plot(epochs, val_bbox_losses, color=bbox_color, linestyle=':', label="Val BBOX Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()


: 

In [ ]:
import os
import random
import cv2
import numpy as np
from PIL import Image
import torch

# Example predict function (as provided)
def predict(image_path, model, device, threshold=0.7):
    # Load image using PIL
    img = Image.open(image_path).convert("RGB")
    orig_img = np.array(img)  # This will be in RGB format
    # Apply transforms (assume transforms is defined globally)
    img_tensor = transforms(img).to(device)
    img_tensor = img_tensor.unsqueeze(0)
    
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()
    scores = output['scores'].cpu().numpy()
    labels = output['labels'].cpu().numpy()
    
    # Filter out detections below threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]
    return orig_img, boxes, scores, labels

# Define your transforms if not already defined.
transforms = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Assume model and device are defined and loaded elsewhere.
# For example:
# device = torch.device("cuda:0")
# model = <your model loaded on device>

# Directory with test images
test_images_dir = "new_dataset/train/images/desk_frame_00013.png"  # change this to your actual test images directory

# Get list of image files (supporting common image extensions)
# image_files = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
# if len(image_files) == 0:
#     print("No image files found in", test_images_dir)
#     exit()

# # Randomly select an image file
# selected_file = random.choice(image_files)
# image_path = os.path.join(test_images_dir, selected_file)
# print("Selected image:", image_path)

# Run the predict function on the selected image
orig_img, boxes, scores, labels = predict(test_images_dir, model, device, threshold=0.7)

# Draw bounding boxes on the original image.
# Note: orig_img is in RGB, convert to BGR for cv2.imshow.
img_bgr = cv2.cvtColor(orig_img, cv2.COLOR_RGB2BGR)

for box, score, label in zip(boxes, scores, labels):
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img_bgr, f"ID:{label} {score:.2f}", (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Display the image
cv2.imshow("Test Image with Bounding Boxes", img_bgr)
cv2.waitKey(0)
cv2.destroyAllWindows()


### Video Detection

In [3]:
import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights

# Define the transform (same as during training)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda")

# Load and modify the detection model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2
)
model.to(device)

# Load saved weights
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Best_Model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

def predict(image_input, model, device, threshold=0.7, nms_threshold=0.3, top_k=1):
    """
    Predict detections on a given image.
    
    Args:
        image_input (str or np.ndarray): If string, treated as file path;
                                           if np.ndarray, treated as an OpenCV BGR image.
        model: The detection model.
        device: Computation device.
        threshold (float): Confidence threshold.
        nms_threshold (float): IoU threshold for non-maximum suppression.
        top_k (int): Number of top detections to display.
    
    Returns:
        orig_img (np.ndarray): The original image in BGR format.
        boxes (np.ndarray): Array of bounding boxes.
        scores (np.ndarray): Detection scores.
        labels (np.ndarray): Detected labels.
        inference_time_ms (float): Inference time in milliseconds.
    """
    # Check input type and convert accordingly
    if isinstance(image_input, np.ndarray):
        # image_input is a frame (BGR)
        pil_img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
        orig_img = image_input.copy()  # keep a copy in BGR for display
    elif isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
        orig_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Unsupported type for image_input. Must be str or np.ndarray.")
    
    # Apply transforms to create a tensor
    img_tensor = transform(pil_img).to(device)
    img_tensor = img_tensor.unsqueeze(0)  # add batch dimension

    # Measure inference time using OpenCV ticks
    start = cv2.getTickCount()
    with torch.no_grad():
        outputs = model(img_tensor)
    end = cv2.getTickCount()
    inference_time_ms = (end - start) / cv2.getTickFrequency() * 1000.0

    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()
    scores = output['scores'].cpu().numpy()
    labels = output['labels'].cpu().numpy()

    # Filter out detections below confidence threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    if len(boxes) > 0:
        # Optionally, apply non-maximum suppression (NMS)
        if nms_threshold > 0:
            boxes_tensor = torch.tensor(boxes, device=device)
            scores_tensor = torch.tensor(scores, device=device)
            labels_tensor = torch.tensor(labels, device=device)
            keep_indices = torchvision.ops.nms(boxes_tensor, scores_tensor, nms_threshold)
            keep_indices = keep_indices.cpu().numpy()
            boxes = boxes_tensor[keep_indices].cpu().numpy()
            scores = scores_tensor[keep_indices].cpu().numpy()
            labels = labels_tensor[keep_indices].cpu().numpy()

        # Select top_k detections based on scores
        k = min(top_k, len(scores))
        sorted_indices = np.argsort(scores)[::-1][:k]
        boxes = boxes[sorted_indices]
        scores = scores[sorted_indices]
        labels = labels[sorted_indices]

    return orig_img, boxes, scores, labels, inference_time_ms

def main():
    # Choose one of your video files
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\3_Mice.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\TestFile_video.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Cohort_1.mp4"
    video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\random_youtube_video.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\desktop.avi"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\desktop2.avi"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\desktop3.avi"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\brown_rats.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
    top_k = 3
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return

    frame_count = 0  # Initialize a frame counter

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1  # Increment the frame counter
        
        # Predict on the current frame
        orig_img, boxes, scores, labels, inf_time = predict(
            frame, model, device, threshold=0.3, nms_threshold=0.1, top_k=top_k)
        print("Labels:", labels)
        print("Inference time (ms):", inf_time)

        # Draw bounding boxes, label text, and centroid red dot on the frame
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(orig_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(orig_img, f"Rat: {score:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            # Compute and draw the centroid as a red dot
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            cv2.circle(orig_img, (cx, cy), 3, (0, 0, 255), -1)
        
        # Draw inference time on the frame
        cv2.putText(orig_img, f"Inference: {inf_time:.1f} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        
        # Display the frame number (e.g., at the top left corner)
        cv2.putText(orig_img, f"Frame: {frame_count}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2)
        
        # Display the frame with predictions
        cv2.imshow("Predictions", orig_img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


Labels: []
Inference time (ms): 27.7192
Labels: []
Inference time (ms): 27.210600000000003
Labels: []
Inference time (ms): 26.6872
Labels: []
Inference time (ms): 26.6629
Labels: []
Inference time (ms): 26.6205
Labels: []
Inference time (ms): 26.8062
Labels: []
Inference time (ms): 28.615499999999997
Labels: []
Inference time (ms): 27.7452
Labels: []
Inference time (ms): 26.8732
Labels: []
Inference time (ms): 31.2207
Labels: []
Inference time (ms): 27.4381
Labels: []
Inference time (ms): 29.7344
Labels: []
Inference time (ms): 27.4623
Labels: []
Inference time (ms): 27.9841
Labels: []
Inference time (ms): 27.4828
Labels: []
Inference time (ms): 27.4032
Labels: []
Inference time (ms): 27.5031
Labels: []
Inference time (ms): 27.6729
Labels: []
Inference time (ms): 27.099600000000002
Labels: []
Inference time (ms): 28.487599999999997
Labels: []
Inference time (ms): 26.912
Labels: []
Inference time (ms): 27.712600000000002
Labels: []
Inference time (ms): 28.1118
Labels: []
Inference time 

# Calculate PARAMs and MACs

In [ ]:
import torch
import torch.nn as nn
from ptflops import get_model_complexity_info

class SSDWrapper(nn.Module):
    def __init__(self, detection_model):
        super().__init__()
        self.model = detection_model

    def forward(self, x):
        # x will be a single Tensor of shape (batch_size=1, 3, H, W)
        # We need to transform it into a list of images for SSDLite.
        return self.model([x[0]])  # pass as a list with one image
import torch
from ptflops import get_model_complexity_info

# 1. Wrap the SSDLite model
wrapped_model = SSDWrapper(model).to(device)
wrapped_model.eval()

# 2. Define the input shape for which you want to measure FLOPs
input_res = (3, 320, 320)  # (channels, height, width)

# 3. Calculate MACs and Params using ptflops
with torch.cuda.device(0):
    macs, params = get_model_complexity_info(
        wrapped_model,
        input_res,
        as_strings=True,           # If True, returns strings like '0.88 GFLOPs'
        print_per_layer_stat=False # If True, prints layer-by-layer stats
    )

print(f"MACs: {macs}")
print(f"Params: {params}")


MACs: 518.9 MMac
Params: 3.71 M


# Convert to C++ to ONNX

In [ ]:
import torch
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLite320_MobileNet_V3_Large_Weights

# 1) Create model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,  # e.g. background + rat
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)

# 2) Load your trained weights
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
state_dict = torch.load(model_path, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

# 3) Force the model to keep the final detection step in the graph
#    so that the ONNX includes bounding boxes, scores, labels.
#    We'll override the model's transform.postprocess to do nothing
#    but we do want to keep `postprocess_detections`.
#    By default, the official code calls it inside `forward()` only if `not training`.
def keep_postprocess_detections(self, head_outputs, anchors, image_sizes):
    # The default ssd.py code for postprocess_detections does the anchor decode, clamp, etc.
    # We'll call the original method:
    from torchvision.models.detection.ssd import SSD
    return SSD.postprocess_detections(self, head_outputs, anchors, image_sizes)

# Attach this override
model.postprocess_detections = keep_postprocess_detections.__get__(model)

# Optionally override transform.postprocess to do *nothing* (so we keep raw image size).
# model.transform.postprocess = lambda detections, image_sizes, orig_image_sizes: detections

# 4) Create a dummy input
dummy_input = torch.randn(1, 3, 320, 320)

# 5) Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    # We want 3 outputs: "boxes", "scores", "labels"
    # but TorchVision returns them in a single list of dict for each image.
    # By default, it might create sequence outputs. Let's see.
    output_names=["boxes", "scores", "labels"],
    opset_version=12,
    # dynamic_axes={
    #     "input": {0: "batch_size", 2: "height", 3: "width"},
    #     "boxes": {1: "num_boxes"},
    #     "scores": {1: "num_boxes"},
    #     "labels": {1: "num_boxes"},
    # }
)
print("Exported SSDLite model with final bounding boxes to model.onnx.")


c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torchvision\ops\boxes.py:166: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torchvision\ops\boxes.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_y = torch.min(boxes_y, torch.tensor(height, dtype=boxes.dtype, device=boxes.device))
c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torchvision\models\detection\transform.py:308: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTe

Exported SSDLite model with final bounding boxes to model.onnx.


In [ ]:
import torch
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights

# Recreate the model architecture
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,  # background and rat
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)

# Load your trained weights
state_dict = torch.load("Model.pth", map_location="cpu")
model.load_state_dict(state_dict)

# Set the model to evaluation mode
model.eval()

# (Optional) If you want to run the conversion on CPU
device = torch.device("cpu")
model.to(device)

# Create a dummy input matching your expected input size (e.g., 1x3x320x320)
dummy_input = torch.randn(1, 3, 320, 320, device=device)

# Export the model to ONNX
torch.onnx.export(
    model, 
    dummy_input, 
    "model.onnx", 
    export_params=True,              # store the trained parameter weights inside the model file
    opset_version=11,                # choose an appropriate opset version
    do_constant_folding=False,        # optimize constant expressions
    input_names=["input"],           # name your input tensor(s)
    output_names=["boxes", "scores", "labels"],        # name your output tensor(s)
    dynamic_axes={
        "input": {0: "batch_size"},   # enable variable batch size
        "output": {0: "batch_size"}
    }
)
print("Model successfully exported to model.onnx")


c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torch\onnx\utils.py:1824: UserWarning: Provided key output for dynamic axes is not a valid input/output name
  warnings.warn(


Model successfully exported to model.onnx


In [ ]:
import cv2
import numpy as np
import time
import onnxruntime as ort

# --------------------------
# 1) Preprocessing
# --------------------------
def preprocess(frame, target_size=(320, 320)):
    """
    Convert BGR frame to normalized RGB tensor of shape (1, 3, H, W).
    """
    # Convert from BGR to RGB
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Resize to (320 x 320)
    img = cv2.resize(img, target_size)
    
    # Convert to float and normalize to [0,1]
    img = img.astype(np.float32) / 255.0
    
    # Normalize using the same mean and std used in training
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std
    
    # Change from HxWxC to CxHxW, then add batch dimension => (1, 3, 320, 320)
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)
    
    return img

# --------------------------
# 2) ONNX Inference
# --------------------------
def predict(frame, session, threshold=0.2, input_size=(320, 320)):
    """
    Run inference on a single frame using an ONNX session.
    Returns the original frame plus boxes, scores, labels, and inference time.
    """
    # Store original dimensions for later box scaling
    orig_h, orig_w = frame.shape[:2]
    
    # Preprocess
    input_tensor = preprocess(frame, target_size=input_size)
    
    # Get input name for the session
    input_name = session.get_inputs()[0].name
    
    # Run inference and measure time
    start = time.time()
    outputs = session.run(None, {input_name: input_tensor})
    inf_time = (time.time() - start) * 1000.0  # ms
    
    # Model outputs (assuming "boxes", "scores", "labels" in that order)
    boxes = outputs[0]   # Shape: (N, 4)
    scores = outputs[1]  # Shape: (N,)
    labels = outputs[2]  # Shape: (N,)
    
    # Filter predictions by threshold
    valid_idx = scores > threshold
    boxes = boxes[valid_idx]
    scores = scores[valid_idx]
    labels = labels[valid_idx]
    
    # --------------------------
    # 3) If we have at least one detection, keep only the best one
    # --------------------------
    if len(scores) > 0:
        best_idx = np.argmax(scores)
        boxes = boxes[best_idx:best_idx+1]
        scores = scores[best_idx:best_idx+1]
        labels = labels[best_idx:best_idx+1]
    else:
        # No detections above threshold, so we return empty arrays
        boxes = np.array([])
        scores = np.array([])
        labels = np.array([])
    
    # --------------------------
    # 4) Scale Boxes Back
    # --------------------------
    # The boxes are in (320x320) coordinates. Scale them to (orig_w x orig_h).
    scale_x = orig_w / float(input_size[0])
    scale_y = orig_h / float(input_size[1])
    
    if boxes.size > 0:
        boxes[:, [0, 2]] *= scale_x
        boxes[:, [1, 3]] *= scale_y
    
    return frame, boxes, scores, labels, inf_time

# --------------------------
# 4) Main Loop
# --------------------------
def main():
    # Create an ONNX Runtime session (CPU Execution Provider here)
    session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
    
    video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run inference
        orig_img, boxes, scores, labels, inf_time = predict(frame, session, threshold=0.2)
        
        # Draw bounding box (if any)
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(orig_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(orig_img, f"Rat: {score:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            # Draw centroid
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            cv2.circle(orig_img, (cx, cy), 3, (0, 0, 255), -1)
        
        # Display inference time
        cv2.putText(orig_img, f"Inference: {inf_time:.1f} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        
        cv2.imshow("Predictions", orig_img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


#Default SSD LITE

In [ ]:
# import cv2
# import torch
# import torchvision
# import torchvision.transforms as T
# from PIL import Image
# import numpy as np
# from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
# from torchvision.transforms.functional import to_tensor, to_pil_image
# from torchvision.utils import draw_bounding_boxes

# # Set up device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Instantiate the pre-trained SSDLite model using COCO weights.
# # This model is trained for 91 COCO classes.
# model = ssdlite320_mobilenet_v3_large(
#     weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1,
#     weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
# )
# model.to(device)
# model.eval()

# # Retrieve COCO category names from the weights metadata.
# # Note: The COCO model's labels are indexed from 1 (i.e., background is index 0).
# coco_categories = SSDLite320_MobileNet_V3_Large_Weights.COCO_V1.meta["categories"]

# # Define a preprocessing transform (matches the training normalization).
# preprocess = T.Compose([
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225])
# ])

# # Open your video file (or use 0 for webcam)
# video_path = "Video/desktop2.avi"  # Replace with your video file path
# cap = cv2.VideoCapture(video_path)

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Convert the frame (BGR) to RGB and create a PIL image.
#     rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     pil_img = Image.fromarray(rgb_frame)

#     # Preprocess the image.
#     input_tensor = preprocess(pil_img).to(device)

#     # Run inference (model expects a list of images).
#     with torch.no_grad():
#         prediction = model([input_tensor])[0]

#     # Filter predictions with a confidence threshold.
#     score_threshold = 0.5
#     boxes = prediction["boxes"]
#     scores = prediction["scores"]
#     labels = prediction["labels"]
#     keep = scores >= score_threshold
#     boxes = boxes[keep]
#     labels = labels[keep]

#     # Convert label indices to names.
#     # COCO categories are indexed starting at 1, so adjust if needed.
#     labels_text = [coco_categories[label.item()-1] for label in labels]

#     # Prepare image for drawing.
#     # Convert the PIL image to a tensor (scaled to [0,255] and as byte tensor).
#     image_for_draw = to_tensor(pil_img).mul(255).byte().to(device)

#     # Draw bounding boxes on the image.
#     drawn_img = draw_bounding_boxes(image_for_draw, boxes=boxes, labels=labels_text,
#                                     colors="red", width=2)
#     drawn_pil = to_pil_image(drawn_img.cpu())

#     # Display the result using OpenCV.
#     cv2.imshow("Object Detection", cv2.cvtColor(np.array(drawn_pil), cv2.COLOR_RGB2BGR))
#     if cv2.waitKey(1) & 0xFF == ord("q"):
#         break

# cap.release()
# cv2.destroyAllWindows()
